# Companies House - Data Import and Construction Script

This script imports the latest **Basic Company Data** snapshot for *live* companies from the [Free Company Data Product (FCDP)](https://download.companieshouse.gov.uk/en_output.html), along with the latest People with Significant Control (PSC) snapshot from [PSC Data Product](https://download.companieshouse.gov.uk/en_pscdata.html).

The fcdp dataset is provided as a csv file containing one row per company. The PSC dataset is provided as a txt file in JSON format and contains multiple records per company.

To create a usable analytical dataset:
- both sources are reduced to a selected set of relevant columns,
- an age based predictor column is feature engineered (company_age_when_acc_due)
- along with the target variable "overdue" (1,0)
- the PSC data is aggregated to give 1 row per company (psc_count, has_corporate_psc, has_foreign_psc, has_sanctioned_psc, recent_psc_change (within last 12 months))
- the datasets are joined together on company number
- then the final dataset is saved in Parquet format for use in further analysis.

As both datasets are large, all the processed outputs are saved locally in Parquet format, allowing other scripts to load the data efficiently without repeating the full import and transformation process.

Encoding of categorical variables will be conducted after some EDA in the next script.

In [1]:
import warnings
warnings.filterwarnings("ignore") # suppress non-critical warnings throughout the script.

# Import today's date to use in the work.
from datetime import date

In [2]:
# Standard library imports
import importlib
import subprocess
import sys

# List required packages
required_packages = [
    "pandas",          # DataFrames, joins, light transforms
    "numpy",           # Numerical ops
    "matplotlib",      # Plotting
    "seaborn",         # Statistical plots
    "sklearn",         # ML
    "statsmodels",     # Statistical modelling
    "duckdb",          # Fast SQL engine
    "pyarrow",         # Parquet / Arrow
    "ydata_profiling", # EDA (optional)
    "ydata-sdk",       # Advanced data profiling, validation, and synthetic data
]

"""
The ensure_packages function below is set up to check whether each package in the above list is installed.
- If installed it prints a confirmation message with a tick.
- If missing it installs the package using pip install. 
"""
def ensure_packages(packages):
    for pkg in packages:
        try:
            importlib.import_module(pkg)
            print(f"✓ {pkg} already installed")
        except ImportError:
            print(f"✗ {pkg} missing — installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Run the package check.
ensure_packages(required_packages)

✓ pandas already installed
✓ numpy already installed
✓ matplotlib already installed
✓ seaborn already installed
✓ sklearn already installed
✓ statsmodels already installed
✓ duckdb already installed
✓ pyarrow already installed


✓ ydata_profiling already installed
✗ ydata-sdk missing — installing...


In [3]:
# Import the libraries required for this work
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [4]:
# ---------------------------------------------------------
# 1. File paths
# ---------------------------------------------------------
# Raw data file locations (originally downloaded from Companies House)
fcdp_path = r"C:\Users\AT355573\Documents\DS_Assignments\Companies House\data_raw\BasicCompanyDataAsOneFile-2026-03-02.csv"
psc_path  = r"C:\Users\AT355573\Documents\DS_Assignments\Companies House\data_raw\persons-with-significant-control-snapshot-2026-03-12.txt"

# Directory where processed Parquet cached filed will be saved
# Parquet is used because it's faster and more efficient to use in the follow-up scripts than the original downloaded files.
data_processed_dir = r"C:\Users\AT355573\Documents\DS_Assignments\Companies House\data_processed"
os.makedirs(data_processed_dir, exist_ok=True) # This line creates the folder if it doesn't already exist.

# Paths for cached Parquet outputs created during the ETL process.
# These allow other scripts to load the data instantly.
fcdp_parquet       = os.path.join(data_processed_dir, "fcdp.parquet")
psc_parquet        = os.path.join(data_processed_dir, "psc.parquet")
fcdp_selected_path = os.path.join(data_processed_dir, "fcdp_selected.parquet")
psc_features_path  = os.path.join(data_processed_dir, "psc_features.parquet")

# Create an in-memory DuckDB connection, this is used for fast SQL queries on large datasets.
con = duckdb.connect()

In [5]:
# ---------------------------------------------------------
# 2. FCDP load (via DuckDB → Parquet → pandas)
# ---------------------------------------------------------

# If a Parquet version of the FCDP data already exists AND it is not empty, then load it directly,
# print a checkpoint label to tell users what is happening.
if os.path.exists(fcdp_parquet) and os.path.getsize(fcdp_parquet) > 0:
    print(f"✓ Loading FCDP from Parquet")
    fcdp = pd.read_parquet(fcdp_parquet)
else:
    # If the file didn't exist or it was empty then create it, but first print a checkpoint label to tell users what is happening.
    print(f"⏳ Converting FCDP CSV → Parquet (first time only)")
    if not os.path.exists(fcdp_path):
        raise FileNotFoundError(f"Source file not found: {fcdp_path}")

    # Use DuckDB to read the csv file and write it to parquet
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_csv_auto('{fcdp_path}')
        )
        TO '{fcdp_parquet}' (FORMAT PARQUET);
    """)

    # Verify that the Parquet file was created successfully.
    if not os.path.exists(fcdp_parquet) or os.path.getsize(fcdp_parquet) == 0:
        raise RuntimeError(f"Failed to create valid FCDP Parquet file")

    # Now load the fcdp parquet file into pandas.
    fcdp = pd.read_parquet(fcdp_parquet)

# Print the number of rows so the user knows that the load was successful
print(f"FCDP rows: {len(fcdp):,}")

✓ Loading FCDP from Parquet
FCDP rows: 5,677,276


In [6]:
# ---------------------------------------------------------
# 3. PSC load (via DuckDB → Parquet)
# ---------------------------------------------------------

# If a Parquet version of the PSC data already exists AND it is not empty, then load it directly.
# print a checkpoint label to tell users what is happening.
if os.path.exists(psc_parquet) and os.path.getsize(psc_parquet) > 0:
    print(f"✓ PSC Parquet already exists (will use DuckDB directly from it)")
else:
    # If the file didn't exist or it was empty then create it, but first print a checkpoint label to tell users what is happening.
    print(f"⏳ Converting PSC NDJSON → Parquet (first time only)")
    if not os.path.exists(psc_path):
        raise FileNotFoundError(f"Source file not found: {psc_path}")

        # Use DuckDB to read the PSC JSON file and write it to parquet
        # Because JSON data is nested we:
        # - sample a large number of rows to infer the schema
        # - union fields by name
        # - ignore malformed rows instead of crashing
        # DuckDB then writes the resulting dataset directly to Parquet.
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_json_auto(
                '{psc_path}',
                sample_size=100000,
                union_by_name=true,
                ignore_errors=true
            )
        )
        TO '{psc_parquet}' (FORMAT PARQUET);
    """)

    # Verify that the Parquet file was created successfully
    if not os.path.exists(psc_parquet) or os.path.getsize(psc_parquet) == 0:
        raise RuntimeError(f"Failed to create valid PSC Parquet file")

# Don't load this dataset into pandas as it is too large and needs aggregating first.
# Print a checkpoint message for user benefit.
print(f"✓ PSC Parquet ready")

✓ PSC Parquet already exists (will use DuckDB directly from it)
✓ PSC Parquet ready


In [7]:
# ---------------------------------------------------------
# 4. FCDP COLUMN SELECTION + STANDARDISATION
# ---------------------------------------------------------
# At this stage, the raw FCDP dataset has been loaded into pandas.
# This step:
#   - shows the user what fields exist
#   - selects only the columns needed for analysis
#   - renames them to clean, consistent snake_case
#   - prepares the dataset for feature engineering (age, dissolved_flag)
# ---------------------------------------------------------

# Print a list of all available columns in the fcdp dataset to help the user see what fields we have
print(f"FCDP columns:")
for col in fcdp.columns:
    print(f" -", col)

# Select a reduced number of fields that will be used in the analysis.
fcdp_selected = fcdp[
    [
        "CompanyNumber",            # Unique id per company; used as key identifier and to join to PSC data
        "IncorporationDate",        # Date company was formed; used for calculating company_age
        "CompanyCategory",          # Legal form of the company (private ltd, LLP, PLC, etc); useful for segmentation
        "Accounts.NextDueDate",     # Accounts due date; used to calculate whether filing is late
        "Accounts.AccountCategory", # Size classification (Micro/small/medium etc); infers company scale
        "SICCode.SicText_1",        # Primary industry classification (3 additional codes are available); sector classification
        "CountryOfOrigin",          # Country where company was originally formed; geographic classification variable
        "RegAddress.Country",       # Registereed office country; may differ from origin and affect risk
    ]
].copy()

# Give column headers sensible names using common standards
fcdp_selected = fcdp_selected.rename(columns={
    "CompanyNumber": "company_number",
    "IncorporationDate": "incorporation_date",
    "CompanyCategory": "company_category",
    "Accounts.NextDueDate": "account_due_date",
    "Accounts.AccountCategory": "accounts_category",
    "SICCode.SicText_1": "industry",
    "CountryOfOrigin": "country_of_origin",
    "RegAddress.Country": "registered_country",
})

# Display the first few rows of the dataset as a preview
fcdp_selected.head()

FCDP columns:
 - CompanyName
 - CompanyNumber
 - RegAddress.CareOf
 - RegAddress.POBox
 - RegAddress.AddressLine1
 - RegAddress.AddressLine2
 - RegAddress.PostTown
 - RegAddress.County
 - RegAddress.Country
 - RegAddress.PostCode
 - CompanyCategory
 - CompanyStatus
 - CountryOfOrigin
 - DissolutionDate
 - IncorporationDate
 - Accounts.AccountRefDay
 - Accounts.AccountRefMonth
 - Accounts.NextDueDate
 - Accounts.LastMadeUpDate
 - Accounts.AccountCategory
 - Returns.NextDueDate
 - Returns.LastMadeUpDate
 - Mortgages.NumMortCharges
 - Mortgages.NumMortOutstanding
 - Mortgages.NumMortPartSatisfied
 - Mortgages.NumMortSatisfied
 - SICCode.SicText_1
 - SICCode.SicText_2
 - SICCode.SicText_3
 - SICCode.SicText_4
 - LimitedPartnerships.NumGenPartners
 - LimitedPartnerships.NumLimPartners
 - URI
 - PreviousName_1.CONDATE
 - PreviousName_1.CompanyName
 - PreviousName_2.CONDATE
 - PreviousName_2.CompanyName
 - PreviousName_3.CONDATE
 - PreviousName_3.CompanyName
 - PreviousName_4.CONDATE
 - Previ

,company_number,incorporation_date,company_category,account_due_date,accounts_category,industry,country_of_origin,registered_country
0,08209948,2012-09-11,Private Limited Company,2026-06-30,DORMANT,99999 - Dormant Company,United Kingdom,ENGLAND
1,11743365,2018-12-28,Private Limited Company,2026-09-30,DORMANT,59112 - Video production activities,United Kingdom,None
2,16873705,2025-11-25,Private Limited Company,2027-08-25,NO ACCOUNTS FILED,86210 - General medical practice activities,United Kingdom,UNITED KINGDOM
3,15073164,2023-08-15,Private Limited Company,2026-12-31,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,United Kingdom,ENGLAND
4,13522064,2021-07-21,Private Limited Company,2027-04-30,MICRO ENTITY,58290 - Other software publishing,United Kingdom,UNITED KINGDOM


In [8]:
# ---------------------------------------------------------
# 5. COMPANY FEATURE ENGINEERING
# ---------------------------------------------------------
# Now that FCDP has been cleaned and standardised, we create
# two core engineered features:
#   - company_age_when_acc_due: years since incorporation
#   - filed_late: target variable (1 or 0)
# ---------------------------------------------------------

# Reference point
today = pd.Timestamp.today().normalize()

# Ensure date columns are proper datetime
fcdp_selected["incorporation_date"] = pd.to_datetime(fcdp_selected["incorporation_date"], errors="coerce")
fcdp_selected["account_due_date"] = pd.to_datetime(fcdp_selected["account_due_date"], errors="coerce")


# 1. Use incorporation date and account_due_date to calculate the company age at the time accounts are due
fcdp_selected["company_age_when_acc_due"] = (
    (fcdp_selected["account_due_date"] - fcdp_selected["incorporation_date"])
    .dt.days / 365.25
)


# 2. Target variable: Overdue
fcdp_selected["overdue"] = (today > fcdp_selected["account_due_date"]).astype(int)

# Remove date columns and company_status column as they are no longer needed.
fcdp_selected = fcdp_selected.drop(columns=["incorporation_date", "account_due_date"])

# Save updated FCDP with engineered features
print(f"✓ Added company_age_when_acc_due and overdue target variable to FCDP dataset")
fcdp_selected.head()

✓ Added company_age_when_acc_due and overdue target variable to FCDP dataset


,company_number,company_category,accounts_category,industry,country_of_origin,registered_country,company_age_when_acc_due,overdue
0,08209948,Private Limited Company,DORMANT,99999 - Dormant Company,United Kingdom,ENGLAND,13.798768,0
1,11743365,Private Limited Company,DORMANT,59112 - Video production activities,United Kingdom,None,7.756331,0
2,16873705,Private Limited Company,NO ACCOUNTS FILED,86210 - General medical practice activities,United Kingdom,UNITED KINGDOM,1.746749,0
3,15073164,Private Limited Company,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,United Kingdom,ENGLAND,3.378508,0
4,13522064,Private Limited Company,MICRO ENTITY,58290 - Other software publishing,United Kingdom,UNITED KINGDOM,5.774127,0


In [9]:
fcdp_selected["overdue"].value_counts().to_frame("count").assign(
    percentage=lambda x: (x["count"] / x["count"].sum() * 100).round(2)
)

,count,percentage
overdue,,
0,5225219,92.04
1,452057,7.96


In [10]:
# ---------------------------------------------------------
# 6. SAVE CLEANED FCDP DATASET (for downstream scripts)
# ---------------------------------------------------------
# At this point:
#   - The raw FCDP file has been loaded
#   - Only the required columns have been selected
#   - Column names have been standardised
#
# This step saves the cleaned FCDP subset to Parquet so that
# future scripts can load it instantly without repeating the
# expensive CSV → pandas transformation.
# ---------------------------------------------------------

fcdp_selected.to_parquet(fcdp_selected_path, index=False)
print(f"✓ Saved cleaned FCDP dataset to {fcdp_selected_path} on {date.today()}")

✓ Saved cleaned FCDP dataset to C:\Users\AT355573\Documents\DS_Assignments\Companies House\data_processed\fcdp_selected.parquet on 2026-03-14


In [11]:
# ---------------------------------------------------------
# 7. PSC STRUCTURE INSPECTION (understand nested JSON fields)
# ---------------------------------------------------------
# PSC data is deeply nested JSON stored inside a 'data' column.
# Before flattening or selecting fields, we inspect a single row
# to understand the top-level keys available inside the PSC object.
# This avoids loading the full dataset and helps guide the flattening step.
# ---------------------------------------------------------

# Load one PSC row to inspect the structure
sample_psc = con.execute(f"""
    SELECT data
    FROM read_parquet('{psc_parquet}')
    LIMIT 1
""").fetchone()[0]

# Print the top-level keys inside the PSC 'data' object
print(f"Top-level PSC data fields:")
for key in sample_psc.keys():
    print(f" -", key)

Top-level PSC data fields:
 - address
 - etag
 - identification
 - kind
 - links
 - name
 - natures_of_control
 - notified_on
 - country_of_residence
 - date_of_birth
 - identity_verification_details
 - name_elements
 - nationality
 - ceased_on
 - ceased
 - description
 - is_sanctioned
 - principal_office_address


In [12]:
# ---------------------------------------------------------
# 8. PSC PROCESSING (flatten → recent flags → aggregate)
# ---------------------------------------------------------
# In this step we:
#   1. Create a DuckDB table over the PSC Parquet
#   2. Flatten the nested JSON structure into simple columns
#   3. Compute recent PSC activity flags (last 12 months)
#   4. Aggregate PSC features to company level
#
# This produces a compact PSC feature table that can be joined
# to the FCDP company dataset in later steps.
# ---------------------------------------------------------

# Step 7.1 — Create a DuckDB table over the PSC Parquet
print(f"🔧 PSC Step 1: Creating DuckDB table from Parquet...")
con.execute(f"""
    CREATE OR REPLACE TABLE psc AS
    SELECT
        company_number,                                          -- Company identifier
        data['kind']::VARCHAR AS kind,                           -- PSC type
        TRY_CAST(data['notified_on'] AS DATE) AS notified_on,    -- PSC added
        TRY_CAST(data['ceased_on'] AS DATE) AS ceased_on,        -- PSC ceased
        CAST(data['is_sanctioned'] AS BOOLEAN) AS is_sanctioned, -- Sanctions flag
        LOWER(data['address']['country']) AS psc_country         -- Country normalised
    FROM read_parquet('{psc_parquet}');
""")
print(f"✓ PSC table created")

# Step 7.2 — Flatten PSC fields + compute recent flags
print(f"🔧 PSC Step 2: Flattening PSC fields...")
con.execute(f"""
    CREATE OR REPLACE TABLE psc_flat AS
    SELECT
        company_number,
        kind,
        notified_on,
        ceased_on,
        is_sanctioned,
        psc_country,

        -- Recent PSC activity (last 12 months)
        (notified_on >= CURRENT_DATE - INTERVAL 12 MONTH) AS recent_added,
        (ceased_on   >= CURRENT_DATE - INTERVAL 12 MONTH) AS recent_ceased

    FROM psc;
""")
print(f"✓ PSC flattening complete")

# Step 7.3 — Aggregate PSC features to company level
print(f"🔧 PSC Step 3: Aggregating PSC features to company level...")
psc_features_df = con.execute(f"""
    SELECT
        company_number,                                 -- One row per company
        COUNT(*) AS psc_count,                          -- Total PSCs

        MAX(CASE WHEN kind = 'corporate-entity-person-with-significant-control'
                 THEN 1 ELSE 0 END) AS has_corporate_psc,

        MAX(CASE WHEN psc_country IS NOT NULL
                  AND psc_country NOT IN ('united kingdom', 'uk')
                 THEN 1 ELSE 0 END) AS has_foreign_psc,

        MAX(CASE WHEN is_sanctioned THEN 1 ELSE 0 END) AS has_sanctioned_psc,

        MAX(CASE WHEN recent_added OR recent_ceased THEN 1 ELSE 0 END)
            AS recent_psc_change

    FROM psc_flat
    GROUP BY company_number;
""").df()
print(f"✓ PSC aggregation complete")

print(f"🔍 Preview of PSC features:")
psc_features_df.head()

🔧 PSC Step 1: Creating DuckDB table from Parquet...
✓ PSC table created
🔧 PSC Step 2: Flattening PSC fields...
✓ PSC flattening complete
🔧 PSC Step 3: Aggregating PSC features to company level...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ PSC aggregation complete
🔍 Preview of PSC features:


,company_number,psc_count,has_corporate_psc,has_foreign_psc,has_sanctioned_psc,recent_psc_change
0,10985847,3,1,0,0,0
1,08070707,1,0,0,0,0
2,07676977,2,0,0,0,0
3,06807536,5,0,1,0,0
4,14970844,1,0,1,0,0


In [13]:
# ---------------------------------------------------------
# 9. SAVE PSC FEATURES (for downstream scripts)
# ---------------------------------------------------------
# At this point:
#   - PSC has been flattened
#   - Recent activity flags have been computed
#   - Company‑level PSC features have been aggregated
#
# This step saves the PSC feature table to Parquet so that
# later notebooks/scripts can load it instantly without
# repeating the expensive PSC processing pipeline.
# ---------------------------------------------------------

psc_features_df.to_parquet(psc_features_path, index=False)
print(f"✓ Saved PSC features to {psc_features_path} on {date.today()}")

✓ Saved PSC features to C:\Users\AT355573\Documents\DS_Assignments\Companies House\data_processed\psc_features.parquet on 2026-03-14


In [14]:
# ---------------------------------------------------------
# 10. JOIN FCDP + PSC FEATURES → FINAL MODELLING DATASET
# ---------------------------------------------------------
# In this step we:
#   - Load the cleaned FCDP dataset (company-level fields)
#   - Load the aggregated PSC feature table (PSC-level signals)
#   - Perform a LEFT JOIN to retain all companies, even those
#     without PSC records (PSC is optional for many firms)
#   - Save the final modelling dataset to Parquet
#
# This produces a unified, analysis-ready dataset combining
# company attributes + PSC risk indicators.
# ---------------------------------------------------------

print(f"🔄 Loading processed FCDP and PSC feature tables...")

# Load the pre-processed datasets from Parquet
fcdp_selected = pd.read_parquet(fcdp_selected_path)
psc_features  = pd.read_parquet(psc_features_path)

print(f"FCDP rows: {len(fcdp_selected):,}")
print(f"PSC feature rows: {len(psc_features):,}")

# ---------------------------------------------------------
# Perform the join
# ---------------------------------------------------------
# Left join keeps all companies from FCDP, even if they have no PSC data.
final_dataset = fcdp_selected.merge(
    psc_features,
    on="company_number",
    how="left"
)

# Reorder columns into a sensible order

def reorder_columns(df):
    ordered_cols = [
        "company_number",           # Unique Companies House ID
        "company_category",         # Company type (Ltd, LLP, etc.)
        "company_age_when_acc_due", # Company age when the next account is due to be filed.
        "accounts_category",        # Accounts filing category (micro, small, full)
        "industry",                 # SIC-derived industry/sector
        "country_of_origin",        # Country where company was incorporated
        "registered_country",       # Country where company is currently registered
        "psc_count",                # Number of PSCs
        "has_corporate_psc",        # True if any PSC is a corporate entity
        "has_foreign_psc",          # True if any PSC is non‑UK
        "has_sanctioned_psc",       # True if any PSC appears on sanctions lists
        "recent_psc_change",        # PSC changes in recent period
        "overdue"                   # Target variable (1 = overdue)
    ]
    return df[ordered_cols]

final_dataset = reorder_columns(final_dataset)

# PSC columns that should be integer end up being float due to the way pandas behaves in the join, so convert them back to integer.
psc_flag_cols = [
    "psc_count",
    "has_corporate_psc",
    "has_foreign_psc",
    "has_sanctioned_psc",
    "recent_psc_change"
]

final_dataset[psc_flag_cols] = (
    final_dataset[psc_flag_cols]
    .fillna(0)      # companies with no PSCs
    .astype(int)    # convert back to integer
)

print(f"✓ Join complete")
print(f"Final dataset rows: {len(final_dataset):,}")
print(f"Final dataset columns: {len(final_dataset.columns)}")

# ---------------------------------------------------------
# Save final modelling dataset
# ---------------------------------------------------------
final_output_path = os.path.join(
    data_processed_dir,
    "company_psc_dataset.parquet"
)

final_dataset.to_parquet(final_output_path, index=False)

print(f"✓ Final modelling dataset saved to:\n  {final_output_path} on {date.today()}")
final_dataset.head()

🔄 Loading processed FCDP and PSC feature tables...
FCDP rows: 5,677,276
PSC feature rows: 10,511,118
✓ Join complete
Final dataset rows: 5,677,276
Final dataset columns: 13
✓ Final modelling dataset saved to:
  C:\Users\AT355573\Documents\DS_Assignments\Companies House\data_processed\company_psc_dataset.parquet on 2026-03-14


,company_number,company_category,company_age_when_acc_due,accounts_category,industry,country_of_origin,registered_country,psc_count,has_corporate_psc,has_foreign_psc,has_sanctioned_psc,recent_psc_change,overdue
0,08209948,Private Limited Company,13.798768,DORMANT,99999 - Dormant Company,United Kingdom,ENGLAND,2,0,1,0,0,0
1,11743365,Private Limited Company,7.756331,DORMANT,59112 - Video production activities,United Kingdom,None,1,0,0,0,0,0
2,16873705,Private Limited Company,1.746749,NO ACCOUNTS FILED,86210 - General medical practice activities,United Kingdom,UNITED KINGDOM,1,0,0,0,1,0
3,15073164,Private Limited Company,3.378508,TOTAL EXEMPTION FULL,70229 - Management consultancy activities othe...,United Kingdom,ENGLAND,1,0,1,0,0,0
4,13522064,Private Limited Company,5.774127,MICRO ENTITY,58290 - Other software publishing,United Kingdom,UNITED KINGDOM,1,0,0,0,0,0
